# Libro–Notebook: Librería para Análisis de Plantas y Diseño de Compensadores

**Tema:** plantas dinámicas, funciones de transferencia, Bode, Nyquist, lugar geométrico de las raíces, parámetros temporales y diseño de compensadores.

**Objetivo:** construir una librería didáctica en Python para analizar plantas y diseñar compensadores de adelanto, atraso y adelanto–atraso.

Este notebook está pensado como un pequeño libro técnico: primero desarrolla la herramienta, luego presenta el manual de uso y finalmente resuelve ejemplos progresivos.

## Índice

1. Prólogo  
2. Presentación conceptual  
3. Requisitos del entorno  
4. Construcción de la librería `ctrl_lab`  
5. Manual de uso  
6. Fundamentos matemáticos mínimos  
7. Análisis básico de plantas  
8. Respuesta temporal y parámetros dinámicos  
9. Bode: magnitud, fase y márgenes  
10. Nyquist  
11. Lugar geométrico de las raíces  
12. Compensadores de adelanto, atraso y adelanto–atraso  
13. Ejemplo 1: planta de primer orden  
14. Ejemplo 2: planta de segundo orden subamortiguada  
15. Ejemplo 3: planta con integrador  
16. Ejemplo 4: compensador de adelanto  
17. Ejemplo 5: compensador de atraso  
18. Ejemplo 6: compensador adelanto–atraso  
19. Ejemplo 7: comparación integral  
20. Criterios prácticos de ingeniería  
21. Conclusiones  
22. Plantilla rápida de trabajo

# 1. Prólogo

En control automático, una planta no es solamente una ecuación: es un sistema dinámico que transforma entradas en salidas.

Para sistemas lineales invariantes en el tiempo, la relación se expresa como:

$$
G(s)=\frac{Y(s)}{X(s)}
$$

Pero el significado físico profundo es temporal:

$$
y(t)=g(t)*x(t)
$$

La salida es la convolución entre la respuesta impulsional de la planta y la entrada. Laplace transforma esa convolución en producto:

$$
\mathcal{L}\{g(t)*x(t)\}=G(s)X(s)
$$

Por eso funciona el álgebra de bloques.  
Por eso los bloques en cascada se multiplican.  
Por eso un compensador puede modificar una planta: porque agrega su propia dinámica.

# 2. Presentación conceptual

Dada una planta:

$$
G_p(s)
$$

se introduce un compensador:

$$
G_c(s)
$$

El lazo abierto compensado queda:

$$
G_{OL}(s)=G_c(s)G_p(s)
$$

Y con realimentación unitaria negativa:

$$
G_{CL}(s)=\frac{G_c(s)G_p(s)}{1+G_c(s)G_p(s)}
$$

Diseñar un compensador significa modificar polos, ceros, ganancia, fase y respuesta temporal para lograr un comportamiento deseado.

# 3. Requisitos del entorno

La librería se construye con:

- `numpy`
- `scipy`
- `matplotlib`

No se requiere `python-control`. La intención es que el código sea transparente, didáctico y modificable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import warnings

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
warnings.filterwarnings("ignore")

# 4. Construcción de la librería `ctrl_lab`

La siguiente celda define una mini-librería propia para:

- crear funciones de transferencia;
- multiplicar, sumar y dividir bloques;
- calcular polos y ceros;
- cerrar lazo con realimentación unitaria;
- graficar escalón, impulso, Bode, Nyquist y root locus;
- estimar parámetros temporales;
- estimar márgenes de ganancia y fase;
- crear compensadores de adelanto, atraso y adelanto–atraso.

In [ ]:
class TF:
    """
    Funcion de transferencia continua SISO:
    G(s) = num(s) / den(s)
    """
    def __init__(self, num, den, name="G"):
        self.num = np.trim_zeros(np.array(num, dtype=float), "f")
        self.den = np.trim_zeros(np.array(den, dtype=float), "f")
        self.name = name

        if len(self.num) == 0:
            self.num = np.array([0.0])
        if len(self.den) == 0:
            raise ValueError("El denominador no puede estar vacio.")

        if abs(self.den[0]) < 1e-15:
            raise ValueError("El primer coeficiente del denominador no puede ser cero.")

        if self.den[0] != 1:
            self.num = self.num / self.den[0]
            self.den = self.den / self.den[0]

    def scipy(self):
        return signal.TransferFunction(self.num, self.den)

    def __repr__(self):
        return f"{self.name}(s) = ({np.poly1d(self.num)}) / ({np.poly1d(self.den)})"

    def copy(self, name=None):
        return TF(self.num.copy(), self.den.copy(), name or self.name)

    def poles(self):
        return np.roots(self.den)

    def zeros(self):
        if len(self.num) == 1 and abs(self.num[0]) < 1e-12:
            return np.array([])
        return np.roots(self.num)

    def dc_gain(self):
        den0 = np.polyval(self.den, 0)
        num0 = np.polyval(self.num, 0)
        if abs(den0) < 1e-15:
            return np.inf
        return num0 / den0

    def eval(self, s):
        return np.polyval(self.num, s) / np.polyval(self.den, s)

    def __mul__(self, other):
        if isinstance(other, TF):
            return TF(
                np.polymul(self.num, other.num),
                np.polymul(self.den, other.den),
                name=f"({self.name}·{other.name})"
            )
        return TF(self.num * other, self.den, name=f"{other}·{self.name}")

    def __rmul__(self, other):
        return self.__mul__(other)

    def __truediv__(self, other):
        if isinstance(other, TF):
            return TF(
                np.polymul(self.num, other.den),
                np.polymul(self.den, other.num),
                name=f"({self.name}/{other.name})"
            )
        return TF(self.num, self.den * other, name=f"({self.name}/{other})")

    def __add__(self, other):
        if isinstance(other, TF):
            num = np.polyadd(np.polymul(self.num, other.den), np.polymul(other.num, self.den))
            den = np.polymul(self.den, other.den)
            return TF(num, den, name=f"({self.name}+{other.name})")
        num = np.polyadd(self.num, other * self.den)
        return TF(num, self.den, name=f"({self.name}+{other})")

    def __sub__(self, other):
        return self + (-1)*other


def feedback(G, H=None, sign=-1, name="T"):
    """
    Realimentacion.
    Para realimentacion negativa unitaria:
    T = G / (1 + G)
    """
    if H is None:
        H = TF([1], [1], name="H")

    GH = G * H

    if sign == -1:
        den_cl = np.polyadd(GH.den, GH.num)
    else:
        den_cl = np.polysub(GH.den, GH.num)

    num_cl = np.polymul(G.num, H.den)
    return TF(num_cl, den_cl, name=name)


def characteristic_poly_open_loop(G):
    """
    Para realimentacion unitaria negativa:
    1 + G(s) = 0
    Si G = N/D, la ecuacion caracteristica es D + N = 0.
    """
    return np.polyadd(G.den, G.num)


def closed_loop_poles(G):
    return np.roots(characteristic_poly_open_loop(G))


def print_tf_data(G):
    print(G)
    print("\nCeros:")
    print(G.zeros())
    print("\nPolos:")
    print(G.poles())
    print("\nGanancia DC:")
    print(G.dc_gain())


def step_response(G, t_end=10, n=1000):
    t = np.linspace(0, t_end, n)
    tout, y = signal.step(G.scipy(), T=t)
    return tout, y


def impulse_response(G, t_end=10, n=1000):
    t = np.linspace(0, t_end, n)
    tout, y = signal.impulse(G.scipy(), T=t)
    return tout, y


def plot_step(G, t_end=10, n=1000, label=None):
    t, y = step_response(G, t_end, n)
    plt.figure()
    plt.plot(t, y, label=label or G.name)
    plt.axhline(y[-1], linestyle="--", linewidth=1)
    plt.title(f"Respuesta al escalon - {G.name}")
    plt.xlabel("Tiempo [s]")
    plt.ylabel("Salida")
    plt.legend()
    plt.show()
    return t, y


def plot_impulse(G, t_end=10, n=1000):
    t, y = impulse_response(G, t_end, n)
    plt.figure()
    plt.plot(t, y, label=G.name)
    plt.title(f"Respuesta al impulso - {G.name}")
    plt.xlabel("Tiempo [s]")
    plt.ylabel("Salida")
    plt.legend()
    plt.show()
    return t, y


def bode_data(G, w=None):
    if w is None:
        w = np.logspace(-3, 3, 2000)
    w, mag, phase = signal.bode(G.scipy(), w=w)
    return w, mag, phase


def plot_bode(G, w=None):
    w, mag, phase = bode_data(G, w)

    plt.figure()
    plt.semilogx(w, mag)
    plt.title(f"Bode - Magnitud - {G.name}")
    plt.xlabel("Frecuencia angular [rad/s]")
    plt.ylabel("Magnitud [dB]")
    plt.show()

    plt.figure()
    plt.semilogx(w, phase)
    plt.title(f"Bode - Fase - {G.name}")
    plt.xlabel("Frecuencia angular [rad/s]")
    plt.ylabel("Fase [grados]")
    plt.show()

    return w, mag, phase


def plot_nyquist(G, w=None):
    if w is None:
        w = np.logspace(-3, 3, 4000)
    s = 1j*w
    Gjw = G.eval(s)

    plt.figure()
    plt.plot(np.real(Gjw), np.imag(Gjw), label="omega > 0")
    plt.plot(np.real(Gjw), -np.imag(Gjw), linestyle="--", label="simetria omega < 0")
    plt.scatter([-1], [0], marker="x", s=100, label="-1 + j0")
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.title(f"Nyquist - {G.name}")
    plt.xlabel("Re{G(jw)}")
    plt.ylabel("Im{G(jw)}")
    plt.axis("equal")
    plt.legend()
    plt.show()
    return Gjw


def root_locus(G, k_values=None, xlim=None, ylim=None):
    """
    Lugar geometrico de las raices para:
    1 + K G(s) = 0
    Si G = N/D, entonces D(s) + K N(s) = 0.
    """
    if k_values is None:
        k_values = np.logspace(-3, 3, 800)

    roots = []
    for K in k_values:
        poly = np.polyadd(G.den, K * G.num)
        roots.append(np.roots(poly))
    roots = np.array(roots, dtype=complex)

    plt.figure()
    for i in range(roots.shape[1]):
        plt.plot(np.real(roots[:, i]), np.imag(roots[:, i]), ".", markersize=2)

    p = G.poles()
    z = G.zeros()

    if len(p):
        plt.scatter(np.real(p), np.imag(p), marker="x", s=100, label="Polos OL")
    if len(z):
        plt.scatter(np.real(z), np.imag(z), marker="o", facecolors="none", s=100, label="Ceros OL")

    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.title(f"Lugar Geometrico de las Raices - {G.name}")
    plt.xlabel("Re{s}")
    plt.ylabel("Im{s}")

    if xlim:
        plt.xlim(xlim)
    if ylim:
        plt.ylim(ylim)

    plt.legend()
    plt.show()
    return k_values, roots


def step_info(t, y, tolerance=0.02):
    """
    Estima parametros temporales ante escalon:
    valor final, pico, sobrepico, tiempo de subida y establecimiento.
    """
    y_final = y[-1]
    y_peak = np.max(y)

    if abs(y_final) < 1e-12:
        overshoot = np.nan
    else:
        overshoot = max(0, (y_peak - y_final) / abs(y_final) * 100)

    def first_cross(level):
        idx = np.where(y >= level)[0]
        return t[idx[0]] if len(idx) else np.nan

    if y_final >= 0:
        t10 = first_cross(0.1*y_final)
        t90 = first_cross(0.9*y_final)
    else:
        t10, t90 = np.nan, np.nan

    rise_time = t90 - t10 if not np.isnan(t10) and not np.isnan(t90) else np.nan

    band = tolerance * max(abs(y_final), 1e-12)
    outside = np.where(np.abs(y - y_final) > band)[0]
    settling_time = t[outside[-1] + 1] if len(outside) and outside[-1] + 1 < len(t) else t[0]

    return {
        "valor_final": float(y_final),
        "valor_pico": float(y_peak),
        "sobrepico_%": float(overshoot),
        "tiempo_subida_10_90": float(rise_time),
        "tiempo_establecimiento": float(settling_time)
    }


def damping_and_natural_frequency_from_poles(poles):
    """
    Estima zeta, wn y wd desde polos complejos dominantes.
    """
    complex_poles = [p for p in poles if abs(np.imag(p)) > 1e-8]
    if not complex_poles:
        return None

    p = complex_poles[0]
    wn = abs(p)
    zeta = -np.real(p) / wn
    wd = abs(np.imag(p))

    return {
        "polo": p,
        "zeta": float(zeta),
        "wn_rad_s": float(wn),
        "wd_rad_s": float(wd)
    }


def margins(G, w=None):
    """
    Estima margen de ganancia y margen de fase numericamente.
    """
    if w is None:
        w = np.logspace(-4, 5, 20000)

    s = 1j*w
    Gjw = G.eval(s)
    mag = np.abs(Gjw)
    phase = np.unwrap(np.angle(Gjw)) * 180/np.pi

    wcg = None
    PM = None
    idx = np.where(np.diff(np.sign(mag - 1)) != 0)[0]
    if len(idx):
        i = idx[0]
        wcg = float(w[i])
        PM = float(180 + phase[i])

    wcp = None
    GM = None
    GM_dB = None
    idx2 = np.where(np.diff(np.sign(phase + 180)) != 0)[0]
    if len(idx2):
        i = idx2[0]
        wcp = float(w[i])
        GM = float(1 / mag[i]) if mag[i] != 0 else np.inf
        GM_dB = float(20*np.log10(GM))

    return {
        "wcg_rad_s": wcg,
        "PM_deg": PM,
        "wcp_rad_s": wcp,
        "GM": GM,
        "GM_dB": GM_dB
    }


def compare_step(systems, t_end=10, n=1000):
    plt.figure()
    data = {}

    for G in systems:
        t, y = step_response(G, t_end, n)
        plt.plot(t, y, label=G.name)
        data[G.name] = step_info(t, y)

    plt.title("Comparacion de respuestas al escalon")
    plt.xlabel("Tiempo [s]")
    plt.ylabel("Salida")
    plt.legend()
    plt.show()

    return data


def compare_bode(systems, w=None):
    if w is None:
        w = np.logspace(-3, 3, 2000)

    plt.figure()
    for G in systems:
        w, mag, phase = bode_data(G, w)
        plt.semilogx(w, mag, label=G.name)
    plt.title("Comparacion Bode - Magnitud")
    plt.xlabel("Frecuencia angular [rad/s]")
    plt.ylabel("Magnitud [dB]")
    plt.legend()
    plt.show()

    plt.figure()
    for G in systems:
        w, mag, phase = bode_data(G, w)
        plt.semilogx(w, phase, label=G.name)
    plt.title("Comparacion Bode - Fase")
    plt.xlabel("Frecuencia angular [rad/s]")
    plt.ylabel("Fase [grados]")
    plt.legend()
    plt.show()


def lead_compensator(z, p, K=1, name="C_lead"):
    """
    Compensador de adelanto:
    C(s)=K(s+z)/(s+p), con z < p.
    Cero en -z y polo en -p.
    """
    if not z < p:
        print("Advertencia: para adelanto normalmente se espera z < p.")
    return TF([K, K*z], [1, p], name=name)


def lag_compensator(z, p, K=1, name="C_lag"):
    """
    Compensador de atraso:
    C(s)=K(s+z)/(s+p), con p < z.
    Cero en -z y polo en -p.
    """
    if not p < z:
        print("Advertencia: para atraso normalmente se espera p < z.")
    return TF([K, K*z], [1, p], name=name)


def lead_lag_compensator(z_lead, p_lead, z_lag, p_lag, K=1, name="C_lead_lag"):
    C1 = lead_compensator(z_lead, p_lead, K=1, name="Lead")
    C2 = lag_compensator(z_lag, p_lag, K=1, name="Lag")
    C = K * C1 * C2
    C.name = name
    return C


def second_order(wn, zeta, K=1, name="G2"):
    """
    Sistema de segundo orden:
    G(s)=K*wn^2/(s^2 + 2*zeta*wn*s + wn^2)
    """
    return TF([K*wn**2], [1, 2*zeta*wn, wn**2], name=name)


def first_order(tau, K=1, name="G1"):
    """
    Sistema de primer orden:
    G(s)=K/(tau*s + 1)
    """
    return TF([K], [tau, 1], name=name)

# 5. Manual de uso

## Crear una planta

```python
G = TF([1], [1, 2, 0], name="Planta")
```

representa:

$$
G(s)=\frac{1}{s(s+2)}
$$

## Obtener polos, ceros y ganancia DC

```python
G.poles()
G.zeros()
G.dc_gain()
```

## Cerrar lazo

```python
T = feedback(G)
```

calcula:

$$
T(s)=\frac{G(s)}{1+G(s)}
$$

## Graficar

```python
plot_step(T)
plot_bode(G)
plot_nyquist(G)
root_locus(G)
```

## Diseñar compensadores

```python
C1 = lead_compensator(z=2, p=12, K=5)
C2 = lag_compensator(z=1, p=0.1, K=1)
C3 = lead_lag_compensator(2, 12, 0.5, 0.05, K=3)
```

# 6. Fundamentos matemáticos mínimos

Si:

$$
G(s)=\frac{N(s)}{D(s)}
$$

con realimentación unitaria negativa:

$$
T(s)=\frac{G(s)}{1+G(s)}
$$

entonces:

$$
T(s)=\frac{N(s)}{D(s)+N(s)}
$$

La ecuación característica es:

$$
D(s)+N(s)=0
$$

Los polos de lazo cerrado son las raíces de esa ecuación. Su ubicación en el plano complejo determina estabilidad, rapidez, amortiguamiento y oscilación.

# 7. Análisis básico de plantas

Una planta de primer orden:

$$
G(s)=\frac{K}{\tau s+1}
$$

tiene un polo en:

$$
s=-\frac{1}{\tau}
$$

Cuanto mayor es $\tau$, más lenta es la respuesta.

In [ ]:
G1 = first_order(tau=2, K=1, name="G1 primer orden tau=2")
print_tf_data(G1)
t, y = plot_step(G1, t_end=12)
print(step_info(t, y))
plot_bode(G1)

# 8. Respuesta temporal y parámetros dinámicos

Sistema de segundo orden:

$$
G(s)=\frac{\omega_n^2}{s^2+2\zeta\omega_n s+\omega_n^2}
$$

- $\omega_n$: frecuencia natural.
- $\zeta$: amortiguamiento.
- Si $0<\zeta<1$, aparecen polos complejos conjugados y oscilación amortiguada.

In [ ]:
G2 = second_order(wn=3, zeta=0.25, K=1, name="G2 segundo orden")
print_tf_data(G2)

t, y = plot_step(G2, t_end=10)
print("Parametros temporales:")
print(step_info(t, y))

print("\nParametros desde polos:")
print(damping_and_natural_frequency_from_poles(G2.poles()))

# 9. Bode: magnitud, fase y márgenes

Bode evalúa:

$$
G(j\omega)
$$

Permite analizar:

- ganancia en baja frecuencia;
- atenuación en alta frecuencia;
- margen de fase;
- margen de ganancia;
- ancho de banda aproximado;
- sensibilidad al ruido.

In [ ]:
plot_bode(G2)
print("Margenes estimados:")
print(margins(G2))

# 10. Nyquist

Nyquist representa la curva $G(j\omega)$ en el plano complejo.

Para realimentación unitaria negativa, el punto crítico es:

$$
-1+j0
$$

La relación de la curva con ese punto permite estudiar estabilidad de lazo cerrado.

In [ ]:
plot_nyquist(G2)

# 11. Lugar geométrico de las raíces

Para:

$$
1+KG(s)=0
$$

si $G(s)=N(s)/D(s)$, entonces:

$$
D(s)+KN(s)=0
$$

El root locus muestra cómo se mueven los polos de lazo cerrado al variar $K$.

In [ ]:
Gp = TF([1], [1, 2, 0], name="Gp = 1/(s(s+2))")
print_tf_data(Gp)
root_locus(Gp, xlim=(-8, 2), ylim=(-5, 5))

# 12. Compensadores

## Adelanto

$$
C_{lead}(s)=K\frac{s+z}{s+p}, \quad z<p
$$

Aporta fase positiva, mejora margen de fase y puede acelerar la respuesta.

## Atraso

$$
C_{lag}(s)=K\frac{s+z}{s+p}, \quad p<z
$$

Aumenta ganancia de baja frecuencia y mejora error estacionario.

## Adelanto–atraso

$$
C(s)=K\frac{(s+z_1)(s+z_2)}{(s+p_1)(s+p_2)}
$$

Permite combinar mejora transitoria y mejora estacionaria.

In [ ]:
Clead_demo = lead_compensator(z=2, p=12, K=5, name="Clead demo")
Clag_demo = lag_compensator(z=1, p=0.1, K=1, name="Clag demo")
Cll_demo = lead_lag_compensator(2, 12, 1, 0.1, K=3, name="C lead-lag demo")

print_tf_data(Clead_demo)
plot_bode(Clead_demo)

print_tf_data(Clag_demo)
plot_bode(Clag_demo)

# 13. Ejemplo 1: planta de primer orden

Planta térmica aproximada:

$$
G_p(s)=\frac{2}{5s+1}
$$

Tiene ganancia estática 2 y constante de tiempo 5 s.

In [ ]:
P1 = first_order(tau=5, K=2, name="Planta termica P1")
print_tf_data(P1)
t, y = plot_step(P1, t_end=30)
print(step_info(t, y))
plot_bode(P1)
plot_nyquist(P1)

# 14. Ejemplo 2: planta de segundo orden subamortiguada

Planta mecánica idealizada:

$$
G_p(s)=\frac{25}{s^2+2(0.2)(5)s+25}
$$

$$
\omega_n=5,\quad \zeta=0.2
$$

In [ ]:
P2 = second_order(wn=5, zeta=0.2, K=1, name="Planta mecanica P2")
print_tf_data(P2)
t, y = plot_step(P2, t_end=8)
print(step_info(t, y))
print(damping_and_natural_frequency_from_poles(P2.poles()))
plot_bode(P2)
root_locus(P2, xlim=(-8, 2), ylim=(-8, 8))

# 15. Ejemplo 3: planta con integrador

$$
G_p(s)=\frac{1}{s(s+2)}
$$

Esta planta posee un integrador. En control, un integrador ayuda al error estacionario, pero también puede reducir amortiguamiento.

In [ ]:
P3 = TF([1], [1, 2, 0], name="P3")
T3 = feedback(P3, name="T3 lazo cerrado")

print("Planta:")
print_tf_data(P3)
print("\nLazo cerrado:")
print_tf_data(T3)

compare_step([T3], t_end=12)
plot_bode(P3)
plot_nyquist(P3)
print("Margenes de lazo abierto P3:")
print(margins(P3))

# 16. Ejemplo 4: compensador de adelanto

Planta:

$$
G_p(s)=\frac{1}{s(s+2)}
$$

Compensador propuesto:

$$
C_{lead}(s)=5\frac{s+2}{s+12}
$$

Objetivo:

- mejorar respuesta transitoria;
- aumentar margen de fase;
- desplazar polos dominantes.

In [ ]:
P = TF([1], [1, 2, 0], name="P")
Clead = lead_compensator(z=2, p=12, K=5, name="Clead")

OL_uncomp = P
CL_uncomp = feedback(OL_uncomp, name="CL sin compensar")

OL_lead = Clead * P
OL_lead.name = "OL con adelanto"
CL_lead = feedback(OL_lead, name="CL con adelanto")

print("Compensador:")
print_tf_data(Clead)

print("\nMargenes sin compensar:")
print(margins(OL_uncomp))
print("\nMargenes con adelanto:")
print(margins(OL_lead))

compare_bode([OL_uncomp, OL_lead])
root_locus(OL_uncomp, xlim=(-15, 2), ylim=(-8, 8))
root_locus(OL_lead, xlim=(-15, 2), ylim=(-8, 8))
compare_step([CL_uncomp, CL_lead], t_end=10)

# 17. Ejemplo 5: compensador de atraso

Planta:

$$
G_p(s)=\frac{1}{s+1}
$$

Compensador:

$$
C_{lag}(s)=\frac{s+1}{s+0.1}
$$

Este compensador aumenta la ganancia de baja frecuencia y mejora el error estacionario.

In [ ]:
P_lag = TF([1], [1, 1], name="P_lag")
Clag = lag_compensator(z=1, p=0.1, K=1, name="Clag")

OL0 = P_lag
CL0 = feedback(OL0, name="CL sin atraso")

OLlag = Clag * P_lag
OLlag.name = "OL con atraso"
CLlag = feedback(OLlag, name="CL con atraso")

print_tf_data(Clag)
compare_bode([OL0, OLlag], w=np.logspace(-3, 2, 2000))
compare_step([CL0, CLlag], t_end=40)

# 18. Ejemplo 6: compensador adelanto–atraso

Planta:

$$
G_p(s)=\frac{1}{s(s+1)}
$$

Compensador:

$$
C(s)=3\frac{(s+2)(s+0.5)}{(s+10)(s+0.05)}
$$

La parte de adelanto mejora transitorio.  
La parte de atraso mejora precisión estacionaria.

In [ ]:
Pll = TF([1], [1, 1, 0], name="Pll")
Cll = lead_lag_compensator(
    z_lead=2, p_lead=10,
    z_lag=0.5, p_lag=0.05,
    K=3,
    name="C adelanto-atraso"
)

OL_base = Pll
CL_base = feedback(OL_base, name="CL base")

OL_ll = Cll * Pll
OL_ll.name = "OL adelanto-atraso"
CL_ll = feedback(OL_ll, name="CL adelanto-atraso")

print_tf_data(Cll)
print("\nMargenes base:")
print(margins(OL_base))
print("\nMargenes compensado:")
print(margins(OL_ll))

compare_bode([OL_base, OL_ll], w=np.logspace(-3, 3, 3000))
root_locus(OL_base, xlim=(-15, 2), ylim=(-10, 10))
root_locus(OL_ll, xlim=(-15, 2), ylim=(-10, 10))
compare_step([CL_base, CL_ll], t_end=30)

# 19. Ejemplo 7: comparación integral

Se comparan estrategias sobre:

$$
G_p(s)=\frac{1}{s(s+2)}
$$

Casos:

1. base;
2. ganancia proporcional;
3. adelanto;
4. atraso;
5. adelanto–atraso.

In [ ]:
Pcmp = TF([1], [1, 2, 0], name="Planta")

Kp = 5 * Pcmp
Kp.name = "OL K=5"

C_lead_cmp = lead_compensator(z=2, p=12, K=5, name="Lead")
OL_lead_cmp = C_lead_cmp * Pcmp
OL_lead_cmp.name = "OL Lead"

C_lag_cmp = lag_compensator(z=0.5, p=0.05, K=1, name="Lag")
OL_lag_cmp = C_lag_cmp * Pcmp
OL_lag_cmp.name = "OL Lag"

C_ll_cmp = lead_lag_compensator(2, 12, 0.5, 0.05, K=3, name="LeadLag")
OL_ll_cmp = C_ll_cmp * Pcmp
OL_ll_cmp.name = "OL LeadLag"

CLs = [
    feedback(Pcmp, name="CL base"),
    feedback(Kp, name="CL K=5"),
    feedback(OL_lead_cmp, name="CL Lead"),
    feedback(OL_lag_cmp, name="CL Lag"),
    feedback(OL_ll_cmp, name="CL LeadLag")
]

compare_bode([Pcmp, Kp, OL_lead_cmp, OL_lag_cmp, OL_ll_cmp], w=np.logspace(-3, 3, 2500))
results = compare_step(CLs, t_end=25)
results

# 20. Criterios prácticos de ingeniería

## Si el sistema es lento

Revisar:

- polos dominantes cercanos al origen;
- bajo ancho de banda;
- tiempo de establecimiento alto.

Acciones posibles:

- aumentar ganancia;
- usar adelanto;
- desplazar polos dominantes hacia la izquierda.

## Si el sistema oscila demasiado

Revisar:

- bajo amortiguamiento $\zeta$;
- margen de fase bajo;
- sobrepico alto.

Acciones posibles:

- aumentar margen de fase;
- compensador de adelanto;
- reducir ganancia si es necesario.

## Si el error estacionario es grande

Revisar:

- ganancia DC;
- tipo del sistema;
- comportamiento cerca de $s=0$.

Acciones posibles:

- atraso;
- acción integral;
- aumento de ganancia de baja frecuencia.

## Si se amplifica ruido

Revisar:

- ganancia en alta frecuencia;
- ancho de banda excesivo;
- acción derivativa agresiva.

Acciones posibles:

- limitar adelanto;
- filtrar derivada;
- reducir ganancia de alta frecuencia.

# 21. Conclusiones

Diseñar compensadores es modificar la estructura dinámica de una planta.

En tiempo:

$$
y(t)=g(t)*x(t)
$$

En Laplace:

$$
Y(s)=G(s)X(s)
$$

En control:

$$
G_{OL}(s)=G_c(s)G_p(s)
$$

Y en lazo cerrado:

$$
G_{CL}(s)=\frac{G_c(s)G_p(s)}{1+G_c(s)G_p(s)}
$$

El compensador modifica polos, ceros, fase, ganancia, respuesta transitoria y precisión estacionaria.

En síntesis:

> compensar una planta es rediseñar su geometría dinámica.

# 22. Plantilla rápida de trabajo

```python
# 1. Definir planta
P = TF([1], [1, 2, 0], name="Planta")

# 2. Analizar planta
print_tf_data(P)
plot_step(P)
plot_bode(P)
plot_nyquist(P)
root_locus(P)

# 3. Diseñar compensador
C = lead_compensator(z=2, p=12, K=5)
# C = lag_compensator(z=1, p=0.1, K=1)
# C = lead_lag_compensator(2, 12, 0.5, 0.05, K=3)

# 4. Lazo abierto compensado
OL = C * P

# 5. Lazo cerrado
CL = feedback(OL)

# 6. Comparar
compare_bode([P, OL])
compare_step([feedback(P), CL])
root_locus(OL)
```

# Anexo: advertencia metodológica

La librería fue diseñada con intención pedagógica. Para uso profesional conviene complementar con:

- discretización;
- saturaciones;
- retardos;
- incertidumbre paramétrica;
- robustez;
- ruido de medición;
- restricciones físicas;
- validación experimental;
- simulación segura antes de aplicar en planta real.